In [6]:
import pandas as pd
from generate_xml import load_data, write_xml
import re

# gloss expansion

In [7]:
import editdistance
import numpy as np

# longest overlapping is longest sequence of characters with 
def get_token_with_longest_overlap(string, list_of_strings):
    lengths = []
    for s in list_of_strings:
        
        if len(string) >= len(s):
            big, small, anchor = string, s, 'string'
        else:
            big, small, anchor = s, string, 's' 
        for i in range(len(big)):
            for j in range(i+1, len(big)+1):
                if big[i:j] in small:
                    # always return s belonging to longest overlapping so if string is حد and list_of_strings is ['ب','حدود'], we return 'حدود'
                    if anchor == 'string':
                        lengths.append((small,big[i:j], j-i))
                    else:
                        lengths.append((big, big[i:j], j-i))
    if lengths:
        max_length = max(lengths, key=lambda x: x[2])[2]
        # catch ties
        if len([x[2] for x in lengths if x[2] == max_length]) > 1:
            return False
        token_with_longest_overlap = max(lengths, key=lambda x: x[2])[0]
        
        return token_with_longest_overlap
    else:
        return None


def get_lemma_pos(manual_tokenization, manual_pos,manual_lemma):
    if not '+' in manual_tokenization:
        return manual_tokenization, manual_pos
    manual_tokenization = manual_tokenization.split('+')
    manual_pos = manual_pos.split('+')
    # first check longest overlapping substring
    longest_overlapping_token = get_token_with_longest_overlap(manual_lemma, manual_tokenization)
    if longest_overlapping_token:
        lemma_pos = manual_pos[manual_tokenization.index(longest_overlapping_token)]
        return longest_overlapping_token, lemma_pos
    else:
        # print(manual_tokenization, manual_pos, manual_lemma)
        distances = [editdistance.distance(tok,manual_lemma) for tok in manual_tokenization]        
        argmin = np.argmin(distances)
        lemma_pos = manual_pos[argmin]
        
        stem = manual_tokenization[argmin]
    return stem,lemma_pos


get_token_with_longest_overlap(string='حدود', list_of_strings=['ب', 'حد'])

'حد'

In [34]:
def expand_gloss_map(gloss_map, secondary_map, max_depth=2, bidirectional_init=True):
    """
    Expand gloss_map with optional bidirectional initialization.
    
    Args:
        gloss_map: {(gloss, pos): set of (lemma, pos)}
        secondary_map: {(lemma, pos): set of (gloss, pos)}
        max_depth: number of expansion iterations
        bidirectional_init: if True, merge secondary_map into gloss_map before expansion
    """
    expanded_map = {k: v.copy() for k, v in gloss_map.items()}
    
    # Initialize bidirectionally so both languages start equal
    if bidirectional_init:
        print("Initializing bidirectional mappings...")
        for lemma_pos, gloss_set in secondary_map.items():
            # Add lemma→gloss mappings
            if lemma_pos not in expanded_map:
                expanded_map[lemma_pos] = set()
            expanded_map[lemma_pos].update(gloss_set)
            
            # Add reverse gloss→lemma mappings
            for gloss_pos in gloss_set:
                if gloss_pos not in expanded_map:
                    expanded_map[gloss_pos] = set()
                expanded_map[gloss_pos].add(lemma_pos)
    
    # Transitive closure
    for depth in range(max_depth):
        print(f"Expansion depth {depth+1}...")
        additions = 0
        snapshot = {k: v.copy() for k, v in expanded_map.items()}
        
        for k, v in snapshot.items():
            original_size = len(expanded_map[k])
            
            for vv in v:
                # Follow the chain
                if vv in snapshot:
                    expanded_map[k].update(snapshot[vv])
            
            additions += len(expanded_map[k]) - original_size
        
        print(f"  Added {additions} new mappings")
        if additions == 0:
            print(f"Converged at depth {depth+1}")
            break
    
    return expanded_map

In [35]:
from copy import deepcopy 
from camel_tools.utils.dediac import dediac_ar
data, metadata = load_data('../../zaebuc_written/ZAEBUC-v2.0_release/')

# get index of arabic entries
isar = data.index.map(lambda x: 'ar' in x[0])

# split available glosses 
data.loc[isar,'expanded_gloss'] = data['gloss'].map(lambda x: [x.strip().lower() for x in re.split(r',|;',x)],na_action='ignore').map(set,na_action='ignore')
data.loc[~isar, 'expanded_gloss'] = [set()]* len(data.loc[~isar])
# get lemma_pos for arabic entries, to limit mappings to lemma + pos mappings combos (only necessary for secondary mapping, but we do it for all to be consistent)
data.loc[isar,['_stem_token','_lemma_pos']] = data.loc[isar].apply(lambda x: get_lemma_pos(x['manual_tokenization'], x['manual_pos'],x['manual_lemma']), axis=1, result_type='expand').to_numpy()
# keep manual pos and lemma for non-arabic entries (because their lemmas allow multitoken, e.g. were+not	AUX+PART	be+not)
data.loc[~isar,'_lemma_pos'] = data.loc[~isar,'manual_pos']
data.loc[~isar,'_lemma'] = data.loc[~isar,'manual_lemma']
# map to diacritized lemma for arabic entries, to be even more specific
data.loc[isar,'_lemma'] = data.loc[isar,'manual_diacritized_lemma']

# add lemma to gloss
data['expanded_gloss'] = data.apply(lambda x: x['expanded_gloss'] | set([x['_lemma']]), axis=1)

# create gloss_map from split gloss+pos to set of lemma+pos, e.g. ('level','NOUN') -> set([('حد','NOUN'), ('مستوى','NOUN')])
expanded_gloss = data.reset_index()[['_lemma','expanded_gloss','_lemma_pos']].explode(['expanded_gloss']).replace('',np.nan).drop_duplicates()
# expanded_gloss.loc[expanded_gloss['expanded_gloss'].isna(), 'expanded_gloss'] = expanded_gloss.loc[expanded_gloss['expanded_gloss'].isna(), '_lemma']
expanded_gloss['_gloss_pos_tuple'] = list(zip(expanded_gloss['expanded_gloss'], expanded_gloss['_lemma_pos']))
expanded_gloss['_lemma_pos_tuple'] = list(zip(expanded_gloss['_lemma'], expanded_gloss['_lemma_pos']))

# Build maps
gloss_map = expanded_gloss.dropna().groupby('_gloss_pos_tuple')[
    '_lemma_pos_tuple'].agg(set).to_dict()
secondary_map = expanded_gloss.dropna().groupby('_lemma_pos_tuple')[
    '_gloss_pos_tuple'].agg(set).to_dict()

# Single call with bidirectional init
gloss_map = expand_gloss_map(
    gloss_map,
    secondary_map,
    max_depth=1,           # Shallow expansion after initialization
    bidirectional_init=True  # Ensures both languages start equal
)

data['_lemma_pos_tuple'] = list(zip(data['_lemma'], data['_lemma_pos']))
data.loc[:, 'expanded_gloss'] = data.loc[:, '_lemma_pos_tuple'].map(
    lambda x: deepcopy(gloss_map.get(x, set())))

data.loc[:,'expanded_gloss'] = data.loc[:,'expanded_gloss'].map(lambda x: set([dediac_ar(t[0]) for t in x]))
# data.loc[~isar,'expanded_gloss'] = data.loc[~isar,'expanded_gloss'].map(lambda x: set([dediac_ar(t[0]) for t in x]))

# map english lemmas to arabic lemmas when english lemma is in gloss of arabic lemma




# all glosses include original entry
# unique_gloss_lemma = data.copy().apply(lambda x: (tuple(x['expanded_gloss']),x['manual_lemma']),axis=1,result_type='expand').drop_duplicates(subset=[0,1]).index
# data.loc[unique_gloss_lemma].apply(lambda x: x['expanded_gloss'].append(x['manual_lemma']), axis=1)
data.apply(lambda x: x['expanded_gloss'].add(x['manual_lemma']), axis=1)
data['expanded_gloss'] = data['expanded_gloss'].map(lambda x: tuple(sorted(x)))
print()

/Users/f/Library/CloudStorage/SynologyDrive-ba3sasah/camelLab/zaebuc/blacklab_zaebuc_corpus/src/generate_xml.py:7: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  en = pd.read_csv(f'{datadir}corrected.analyzed_en.tsv',sep='\t',index_col=[0,2,1])


Initializing bidirectional mappings...
Expansion depth 1...
  Added 45753 new mappings



In [38]:
# test queries:
# - query 'level' and retrieve english entries whose lemma is in the gloss of arabic entries with 'level' in their gloss
# - query 'قدر' and retrieve arabic entries whose gloss includes english entries with 'قدر' in their gloss

data[data['expanded_gloss'].apply(lambda x: 'level' in x)].drop_duplicates(subset=['manual_lemma', 'manual_pos'])['manual_lemma'].values

array(['degree', 'limit', 'category', 'end', 'grade', 'level', 'value',
       'class', 'amount', 'extent', 'plane', 'layer', 'standard', 'rank',
       'border', 'level', 'حد', 'حد', 'درجة', 'درجة', 'قدر', 'درجة', 'حد',
       'صعيد', 'مستوى', 'طبقة', 'مستوى', 'حد', 'مرتبة', 'مقدار', 'قدر'],
      dtype=object)

In [39]:
data[data['expanded_gloss'].apply(lambda x: 'حد' in x)].drop_duplicates(subset=['_lemma_pos_tuple'])['manual_lemma'].values

array(['limit', 'end', 'level', 'extent', 'border', 'حد', 'نهاية', 'مدى',
       'توقف', 'حصر', 'توقف', 'محطة', 'آخر', 'درجة', 'قدر', 'صعيد',
       'مستوى', 'طبقة', 'حد', 'نطاق', 'مرتبة', 'مقدار', 'امتداد'],
      dtype=object)

In [40]:
gloss_map_counts = pd.DataFrame(gloss_map.items(), columns=['gloss','manual_diacritized_lemma']).set_index('gloss')
gloss_map_counts['len'] = gloss_map_counts['manual_diacritized_lemma'].map(len)
# gloss_map_counts[gloss_map_counts['len']>100]

In [41]:
gloss_map_counts.sort_values('len',ascending=False)

,manual_diacritized_lemma,len
gloss,,
"(manner, NOUN)","{(صُورَة, NOUN), (forms, NOUN), (towards, NOUN...",43
"(level, NOUN)","{(amounts, NOUN), (دَرَجَة, NOUN), (extent, NO...",42
"(individuals, NOUN)","{(factor, NOUN), (side, NOUN), (personalities,...",38
"(time, NOUN)","{(clocks, NOUN), (حِقْبَة, NOUN), (sometimes, ...",37
"(offices, NOUN)","{(office, NOUN), (district, NOUN), (side, NOUN...",35
...,...,...
"(optimistic, ADJ)","{(optimistic, ADJ)}",1
"(opt, VERB)","{(opt, VERB)}",1
"(oppose, VERB)","{(oppose, VERB)}",1


In [51]:
gloss_map_counts['len'].value_counts().sort_index()


len
1     2998
2      856
3     1991
4     1568
5     1793
6      916
7     1057
8      644
9      583
10     417
11     311
12     234
13     202
14     129
15     142
16      89
17      63
18      57
19      50
20      39
21      39
22      26
23      22
24      16
25      14
26      10
27      14
28      13
29       5
30       5
31       1
32       3
33       2
34       4
35       2
37       1
38       1
42       1
43       1
Name: count, dtype: int64

# write xml

In [50]:
write_xml(data, metadata, out_path="../data/zaebuc_written.xml", sample=None)
write_xml(data, metadata, out_path="../data/zaebuc_written_sample.xml", sample=100)

In [20]:
list(data.columns)

['word',
 'flag',
 'auto_tokenization',
 'auto_pos',
 'auto_lemma',
 'manual_tokenization',
 'manual_pos',
 'manual_lemma',
 'comment',
 'manual_diacritized_lemma',
 'gloss',
 'core_pgn',
 'pron_pgn',
 'expanded_gloss',
 '_stem_token',
 '_lemma_pos',
 '_lemma',
 '_lemma_pos_tuple']